This vignette is a hands-on guide to epmcminer, a desktop app that searches [Europe PMC](https://europepmc.org) and batch-downloads open-access PDFs into a structured local folder. It walks through all four wizard screens with step-by-step instructions and ends with a worked example showing the full output.

## The scenario

Maria is a postdoctoral researcher writing a systematic review on *neuroplasticity rehabilitation*. She needs 50 CC-BY open-access papers — fast. Downloading them one by one from PubMed would take the better part of an afternoon. With epmcminer she's done in under 10 minutes.

### What epmcminer does

epmcminer is a desktop app that searches the [Europe PubMed Central (Europe PMC)](https://europepmc.org) API, previews matching papers, and downloads their PDFs and metadata into a tidy local folder — no coding required.

### Dependencies

| Requirement | Version |
|---|---|
| Python | 3.12+ |
| PyQt6 | 6.x |
| requests | 2.x |
| pandas | 2.x |
| openpyxl | 3.x |
| reportlab | 4.x |

### Install

```bash
git clone https://github.com/Programming-The-Next-Step-2026/epmcminer.git
cd epmcminer
python -m venv .venv && source .venv/bin/activate
pip install -e .
epmcminer
```

### Maria's workflow

1. She opens epmcminer and types `neuroplasticity AND rehabilitation` into the search box.
2. She keeps the default publication types (Review, Systematic review, etc.), leaves the licence set to **CC-BY**, and sets the date range from 2019-12-31 until today.
3. She clicks **Continue to preview**. The app queries Europe PMC and shows the top 10 results. She switches sort to **Citations** — most-cited papers first.
4. She selects the desired amount of PDFs by setting **Count = 50** and picks her output folder, then clicks **Start download**.
5. The app fetches PDFs in parallel across multiple threads. A live log shows each paper's status as it completes.
6. On the Summary screen she sees 43 downloaded, 7 skipped (PDF unavailable), and exports the table as Excel for her supervisor.

## How it fits together

The four screens map directly onto the service layer. All long-running calls — preview, download, and export — run in a `QThread` worker so the GUI stays responsive.

```mermaid
flowchart TD
    S1[Screen 1 - Search] -->|Continue to preview| PW[PreviewWorker]
    PW -->|SearchService.preview| API[Europe PMC API]
    API -->|top 10 results + hit count| S2[Screen 2 - Preview]
    S2 -->|Start download| DW[DownloadWorker]
    DW -->|DownloadService.download| API
    DW -->|progress per paper| S3[Screen 3 - Download]
    DW -->|ReportService.save_csv| CSV[report.csv]
    S3 -->|download complete| S4[Screen 4 - Summary]
    S4 -->|Export| EW[ExportWorker]
    EW -->|ReportService.export| OUT[report.xlsx or .pdf]
```

## Screen 1 — Search

![Search screen](screenshots/mockup_screen_1.png)

This is where you define what you're looking for.

- **Search query**: type free text — e.g. `neuroplasticity AND rehabilitation`. If you omit AND/OR operators the app inserts AND between every word automatically.
- **Publication types**: pre-filled with 16 types (Review, Systematic review, Clinical trial, etc.). Remove any you don't need by clicking the × on a tag, or add new ones from the dropdown. At least one type must remain selected.
- **License**: defaults to CC-BY. You can add CC-BY-NC, CC0, and others from the dropdown. At least one licence must stay selected.
- **Author ORCIDs** (optional): add one or more ORCID identifiers to restrict results to specific authors. Multiple ORCIDs use OR logic. Leave empty to search all authors.
- **Date range**: defaults to the past five years up to today. Click either field to open a calendar picker. The end date is capped at today.

> the **Continue to preview** button stays disabled until the query box is non-empty and at least one publication type is selected. If it's greyed out, check those two fields first.

## Screen 2 — Preview

![Preview screen](screenshots/mockup_screen_2.png)

Once you click **Continue to preview**, a background thread fires off the API query. You'll see a brief loading animation, then the results land.

- **Stat cards**: three numbers across the top — *Total results* (everything matching your query in the full database), *PDF available* (papers in the preview set with a direct PDF link), and *Previewing* (always 10 — the top results for the active sort order).
- **Sort order**: the dropdown in the results-list header lets you switch between **Relevance** (default), **Date** (newest first), and **Citations** (most cited first). Changing it re-queries the API and refreshes the list live.
- **Download settings**: set the **Count** (how many PDFs you want) and confirm the **Output folder**.
- Click **← Back** at any time to return to Screen 1 with all your filters intact.

> if Total results is 0, head back and broaden the query — try fewer AND clauses, a wider date range, or extra publication types.
>
> the **Start download** button stays disabled until Count ≥ 1 and an output folder is set.

## Screen 3 — Download

![Download screen](screenshots/mockup_screen_3.png)

Click **Start download** and the app hands off to a background worker running multiple parallel download threads.

- **Progress bar**: shows percentage and a running tally — e.g. `46% · 23 of 50 downloaded` — plus an estimated time remaining.
- **Live log**: every paper gets a row the moment it finishes — a green ✓ for a saved PDF, an orange – for a skip, or a red ✗ for a failure. Each row shows the filename and file size (for downloads) or the skip reason.
- **Skip reasons you'll commonly see**:
  - *PDF unavailable* — the paper is open-access but has no direct PDF link in the API response.
  - *Already downloaded* — a file with the same name already exists in your output folder. Safe to re-run into the same folder: existing papers are skipped, not duplicated.
- **Cancel**: click **✕ Cancel** to stop after the current batch. Already-downloaded files are kept. It might take a few moments for the cancellation to go through, as all running threads have to stop beforehand.
- `report.csv` is saved automatically to your output folder the moment the download finishes or is cancelled.

> if most papers show *PDF unavailable*, try switching sort to **Date** or **Relevance**. Highly-cited older papers are sometimes open-access in the metadata but missing a direct PDF link.

## Screen 4 — Summary

![Summary screen](screenshots/mockup_screen_4.png)

The summary screen gives you a quick debrief and lets you export the full results table.

- **Stat cards**: *Downloaded* (PDFs saved to disk), *Skipped* (unavailable or already downloaded), and *Total results* (the full hit count from the API).
- **Search parameters**: a compact recap of your query, sort order, and date range — handy for copying into a methods section.
- **Skipped papers**: if any papers were skipped or failed, they're listed here with title, authors, and the reason. If the skipped count is high, consider relaxing your filters or running a broader search.
- **Export Excel**: opens a save dialog and writes an `.xlsx` file — same columns as `report.csv` but with auto-formatted cells.
- **Export PDF**: writes a landscape-format PDF table — good for attaching to a report or sharing with a supervisor.
- **＋ New search**: resets the wizard to Screen 1 so you can run a different query.

> the export runs in the background. Don't close the app right after clicking Export — wait for the success dialog.

## Example run — "ADHD cognitive training"

Here's what a typical run looks like end to end.

**Settings used**: query `ADHD cognitive training`, licence CC-BY, publication types Review + Systematic review, sort by Citations, Count = 20, date range 2019-01-01 until 2024-12-31.

### Preview results (top 5 of the 10 shown)

| Title | Authors | Year | Journal |
|---|---|---|---|
| Cognitive training in children with ADHD: a systematic review | Cortese S et al. | 2023 | J Child Psychol Psychiatry |
| Neuroplasticity-based interventions for ADHD | Sonuga-Barke E et al. | 2022 | The Lancet Psychiatry |
| Working memory training for ADHD: an updated meta-analysis | Melby-Lervåg M et al. | 2021 | Psychological Bulletin |
| Digital cognitive training in adult ADHD | Kiani B et al. | 2023 | Frontiers in Psychiatry |
| Executive function training across the lifespan in ADHD | Willcutt E et al. | 2022 | Neurosci Biobehav Rev |

### Output folder structure

After the download finishes, the output folder looks like this:

```
~/papers/adhd_search/
├── pdfs/
│   ├── 10.1111_jcpp.13842_Cognitive_training_in_children_with_ADHD.pdf
│   ├── 10.1016_S2215-0366-22-00291-5_Neuroplasticity-based_interventions.pdf
│   ├── 10.1037_bul0000376_Working_memory_training_meta-analysis.pdf
│   ├── 10.3389_fpsyt.2023.1201447_Digital_cognitive_training_adult_ADHD.pdf
│   └── ... (13 more PDFs; 3 skipped — PDF unavailable)
├── logs/
│   └── 2024-11-14_09-22.log
└── report.csv
```

Filenames follow the pattern `{doi}_{title}.pdf` with special characters replaced by underscores. If you re-run the same search into the same folder, existing files are skipped automatically.

### report.csv (first three rows)

The report has one row per paper processed — downloads and skips alike. Columns match those defined in `ReportService`.

```
title,authors,journal,year,doi,status,reason,file_path,query,sort_order,date_from,date_to,licenses,publication_types
Cognitive training in children with ADHD,Cortese S et al.,J Child Psychol Psychiatry,2023,10.1111/jcpp.13842,downloaded,,~/papers/adhd_search/pdfs/10.1111_jcpp.13842_Cognitive_training.pdf,ADHD cognitive training,citations,2019-01-01,2024-12-31,CC-BY,"Review, Systematic review"
Neuroplasticity-based interventions for ADHD,Sonuga-Barke E et al.,The Lancet Psychiatry,2022,10.1016/S2215-0366-22-00291-5,downloaded,,~/papers/adhd_search/pdfs/10.1016_S2215-0366_Neuroplasticity.pdf,ADHD cognitive training,citations,2019-01-01,2024-12-31,CC-BY,"Review, Systematic review"
Working memory training for ADHD,Melby-Lervåg M et al.,Psychological Bulletin,2021,10.1037/bul0000376,skipped,PDF unavailable,,ADHD cognitive training,citations,2019-01-01,2024-12-31,CC-BY,"Review, Systematic review"
```

Skipped papers have a non-empty `reason` and an empty `file_path`. Failed downloads get an HTTP status code or `Connection error` as the reason.